<a href="https://colab.research.google.com/github/lenmecc/miniature-enigma/blob/main/Inventory_optimization_V2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install xlsxwriter

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 14.3 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import datetime
from datetime import timedelta
import os

# ==============================================================================
# 1. CONFIGURACIÓN Y CARGA DE DATOS REALES
# ==============================================================================
def cargar_datos_reales():
    print("--- INICIANDO PROCESO DE CARGA ---")

    # 1.1 Cargar Histórico (Demanda Pasada)
    # Columnas esperadas: DemandaEPS, Planta, Fecha, Placa, Peso
    try:
        print("Cargando Histórico...")
        df_hist = pd.read_excel('Historico.xlsx')
        # Normalizar nombres
        if 'Peso' in df_hist.columns:
            df_hist.rename(columns={'Peso': 'Kilos'}, inplace=True)
        # Asegurar formato Fecha
        df_hist['Fecha'] = pd.to_datetime(df_hist['Fecha'])
    except Exception as e:
        print(f"ERROR cargando Historico.xlsx: {e}")
        return None, None, None

    # 1.2 Cargar Inventario (Stock Actual)
    # Columnas esperadas: Placa, Peso total inventario
    try:
        print("Cargando Inventario...")
        df_inv = pd.read_excel('placa disponible_wk30.xlsx')

        # Validación: Si no existe columna 'En_Transito', asumimos 0
        if 'En_Transito' not in df_inv.columns:
            df_inv['En_Transito'] = 0

        # Limpieza básica
        df_inv['Placa'] = df_inv['Placa'].astype(str).str.strip()
    except Exception as e:
        print(f"ERROR cargando Inventario.xlsx: {e}")
        return None, None, None

    # 1.3 Cargar Reporte ERP (Futuro)
    # Columnas esperadas: OrdenEPS, Fecha, Kilos, Placa...
    try:
        print("Cargando Reporte ERP...")
        # NOTA: header=1 salta la primera fila automáticamente (tu paso manual)
        df_erp = pd.read_excel('CapacidadDetalle_wk30.xlsx', header=1)

        # Normalizar nombres si varían
        # Ajusta aquí si tus columnas reales tienen nombres distintos
        #mapa_cols = {
        #    'Peso': 'Kilos',
         #   'Fecha Entrega': 'Fecha' # Ajustar según tu Excel real
       # }
       # df_erp.rename(columns=mapa_cols, inplace=True)

        # Limpieza de Fechas del ERP (A veces vienen vacías o con texto)
        df_erp['Fecha'] = pd.to_datetime(df_erp['Fecha'], errors='coerce')
        df_erp = df_erp.dropna(subset=['Fecha']) # Eliminar filas sin Fecha válida

    except Exception as e:
        print(f"ERROR cargando Reporte_ERP.xlsx: {e}")
        return None, None, None

    print("¡Datos cargados correctamente!")
    return df_hist, df_inv, df_erp

# ==============================================================================
# 2. LIMPIEZA DE OUTLIERS (WINSORIZACIÓN)
# ==============================================================================
def limpiar_y_auditar_outliers(df_hist_semanal, nivel_confianza=0.95):
    """
    Recorta el top 5% de demandas extremas para no inflar el stock de seguridad.
    """
    df_clean = df_hist_semanal.copy()

    # Calcular el límite (Techo) por Placa
    limites = df_clean.groupby('Placa')['Kilos'].quantile(nivel_confianza).reset_index()
    limites.rename(columns={'Kilos': 'Limite_Winsor'}, inplace=True)

    df_clean = pd.merge(df_clean, limites, on='Placa', how='left')

    # Detectar y registrar
    mask_outlier = df_clean['Kilos'] > df_clean['Limite_Winsor']
    df_log_outliers = df_clean[mask_outlier].copy()

    if not df_log_outliers.empty:
        df_log_outliers['Kilos_Original'] = df_log_outliers['Kilos']
        df_log_outliers['Kilos_Ajustado'] = df_log_outliers['Limite_Winsor']
        df_log_outliers['Recorte_Kg'] = df_log_outliers['Kilos_Original'] - df_log_outliers['Kilos_Ajustado']

    # Aplicar corrección
    df_clean['Kilos'] = np.where(mask_outlier, df_clean['Limite_Winsor'], df_clean['Kilos'])

    # Conteo para resumen
    conteo = df_log_outliers.groupby('Placa').size().reset_index(name='Num_Outliers_Ajustados')

    return df_clean, df_log_outliers, conteo

# ==============================================================================
# 3. MOTOR PRINCIPAL DE CÁLCULO
# ==============================================================================
def motor_abastecimiento(df_hist, df_inv, df_erp):
    print("Ejecutando algoritmos de optimización...")

    # --- A. Preprocesamiento Histórico ---
    # Agrupar historial por semana (Suma de kilos)
    hist_semanal = df_hist.groupby(['Placa', pd.Grouper(key='Fecha', freq='W')])['Kilos'].sum().reset_index()

    # Limpieza de Outliers
    hist_limpio, log_outliers, conteo_outliers = limpiar_y_auditar_outliers(hist_semanal)

    # --- B. Parametrización (Estadísticas) ---
    stats = hist_limpio.groupby('Placa')['Kilos'].agg(['mean', 'std']).reset_index()
    stats.rename(columns={'mean': 'Promedio_Semanal', 'std': 'Desviacion_Std'}, inplace=True)

    stats['ADU'] = stats['Promedio_Semanal'] / 7 # Average Daily Usage
    stats['CV'] = stats['Desviacion_Std'] / stats['Promedio_Semanal'] # Coeficiente Variación
    stats.fillna(0, inplace=True)

    # Definición de Buffers (DDMRP)
    LEAD_TIME = 14 # Días (Ajustar si tienes una columna de LT en inventario)

    stats['Zona_Amarilla'] = stats['ADU'] * LEAD_TIME
    stats['Factor_Seguridad'] = np.where(stats['CV'] > 0.5, 0.8, 0.4)
    stats['Zona_Roja'] = stats['Zona_Amarilla'] * stats['Factor_Seguridad']
    stats['Zona_Verde'] = stats['ADU'] * 7 # Ciclo semanal

    stats['Top_Of_Green'] = stats['Zona_Roja'] + stats['Zona_Amarilla'] + stats['Zona_Verde']
    stats['Punto_Reorden'] = stats['Zona_Roja'] + stats['Zona_Amarilla']

    # --- C. Demanda Futura (Visibilidad Semanal) ---
    today = pd.Timestamp.now().normalize()
    limits = [today + timedelta(days=d) for d in [10, 17, 24]]

    def bucket(f):
        if f < today: return 'Req_Sem_Actual' # Backlog
        if f <= limits[0]: return 'Req_Sem_Actual'
        if f <= limits[1]: return 'Req_Sem_Mas_1'
        if f <= limits[2]: return 'Req_Sem_Mas_2'
        return 'Futuro_Lejano'

    df_erp['Bucket'] = df_erp['Fecha'].apply(bucket)

    visibilidad = df_erp.pivot_table(
        index='Placa', columns='Bucket', values='Kilos', aggfunc='sum', fill_value=0
    ).reset_index()

    # Asegurar que existan las columnas
    for col in ['Req_Sem_Actual', 'Req_Sem_Mas_1', 'Req_Sem_Mas_2']:
        if col not in visibilidad.columns: visibilidad[col] = 0

    # --- D. Detección de Spikes Futuros ---
    demanda_q = []
    horizonte = today + timedelta(days=LEAD_TIME)

    # Solo iteramos sobre placas que existen en el histórico o en el ERP
    placas_activas = set(stats['Placa']).union(set(df_erp['Placa']))

    for placa in placas_activas:
        # Obtener umbral (si es placa nueva sin historia, usamos 0)
        try:
            umbral = stats[stats['Placa']==placa]['Zona_Amarilla'].values[0] * 0.5
        except:
            umbral = 0

        # Buscar órdenes en el horizonte de Lead Time
        ordenes = df_erp[(df_erp['Placa']==placa) & (df_erp['Fecha'] > today) & (df_erp['Fecha'] <= horizonte)]

        spikes = 0
        if not ordenes.empty and umbral > 0:
            diario = ordenes.groupby('Fecha')['Kilos'].sum()
            spikes = diario[diario > umbral].sum()

        demanda_q.append({'Placa': placa, 'Spikes_Futuros': spikes})

    df_spikes = pd.DataFrame(demanda_q)

    # --- E. Consolidación Final ---
    # Base: Inventario (para no perder placas que tengo pero no se piden)
    master = pd.merge(df_inv[['Placa', 'Peso total Inventario', 'En_Transito']], stats, on='Placa', how='outer').fillna(0)
    master = pd.merge(master, visibilidad, on='Placa', how='left').fillna(0)
    master = pd.merge(master, df_spikes, on='Placa', how='left').fillna(0)

    if not conteo_outliers.empty:
        master = pd.merge(master, conteo_outliers, on='Placa', how='left').fillna(0)
    else:
        master['Num_Outliers_Ajustados'] = 0

    # Cálculos Finales de Abastecimiento
    master['Demanda_Total_Calculo'] = master['Req_Sem_Actual'] + master['Spikes_Futuros']

    master['Flujo_Neto'] = (master['Peso total Inventario'] + master['En_Transito']) - master['Demanda_Total_Calculo']

    # Lógica de decisión
    # Si Flujo Neto < Punto Reorden -> COMPRAR
    master['Accion'] = np.where(master['Flujo_Neto'] < master['Punto_Reorden'], 'COMPRAR', 'OK')

    # Cantidad: Llenar hasta el tope (Top of Green)
    master['Sugerencia_Pedido_Kg'] = np.where(
        master['Accion'] == 'COMPRAR',
        master['Top_Of_Green'] - master['Flujo_Neto'],
        0
    )

    # Faltante Inmediato (Para colorear en rojo)
    master['Faltante_Inmediato'] = (master['Req_Sem_Actual'] - master['Peso total Inventario']).clip(lower=0)

    return master, log_outliers

# ==============================================================================
# 4. EXPORTACIÓN A EXCEL
# ==============================================================================
def generar_excel_completo(df_master, df_log_outliers):
    if df_master is None: return

    Fecha_str = pd.Timestamp.now().strftime('%Y-%m-%d')
    nombre_archivo = f"Plan_Abastecimiento_{Fecha_str}.xlsx"

    print(f"Generando archivo: {nombre_archivo} ...")

    try:
        writer = pd.ExcelWriter(nombre_archivo, engine='xlsxwriter')
        workbook = writer.book

        # --- HOJA 1: PARA COMPRAS (Resumen Ejecutivo) ---
        cols_compras = ['Placa', 'Peso total Inventario', 'Req_Sem_Actual',
                        'Req_Sem_Mas_1', 'Req_Sem_Mas_2',
                        'Sugerencia_Pedido_Kg', 'Faltante_Inmediato']

        # Filtramos columnas que existan (por seguridad)
        cols_validas = [c for c in cols_compras if c in df_master.columns]
        df_compras = df_master[cols_validas].copy()

        df_compras.rename(columns={
            'Peso total Inventario': 'Inventario Disp.',
            'Sugerencia_Pedido_Kg': 'CANTIDAD SUGERIDA',
            'Faltante_Inmediato': 'URGENCIA (Faltante Ya)'
        }, inplace=True)

        df_compras.to_excel(writer, sheet_name='Resumen Compras', index=False)

        # Formatos Condicionales
        ws_c = writer.sheets['Resumen Compras']
        fmt_rojo = workbook.add_format({'bg_color': '#FFC7CE', 'font_color': '#9C0006', 'bold': True})
        fmt_amarillo = workbook.add_format({'bg_color': '#FFEB9C', 'font_color': '#9C6500', 'bold': True})
        fmt_header = workbook.add_format({'bold': True, 'bg_color': '#D7E4BC', 'border': 1})

        # Aplicar headers
        for col_num, value in enumerate(df_compras.columns.values):
            ws_c.write(0, col_num, value, fmt_header)

        # Columna Urgencia (Rojo) - Usamos letras fijas asumiendo orden, o calculamos indice
        idx_urgencia = list(df_compras.columns).index('URGENCIA (Faltante Ya)')
        idx_sugerencia = list(df_compras.columns).index('CANTIDAD SUGERIDA')

        # Aplicar a filas de datos
        ws_c.conditional_format(1, idx_urgencia, len(df_compras), idx_urgencia,
                               {'type': 'cell', 'criteria': '>', 'value': 0, 'format': fmt_rojo})
        ws_c.conditional_format(1, idx_sugerencia, len(df_compras), idx_sugerencia,
                               {'type': 'cell', 'criteria': '>', 'value': 1, 'format': fmt_amarillo})

        ws_c.set_column(0, 0, 25) # Ancho Placa
        ws_c.set_column(1, 6, 15) # Ancho Números

        # --- HOJA 2: ANÁLISIS EXPERTO (Detalle Técnico) ---
        # Ordenamos columnas para que sea legible
        cols_prioridad = ['Placa', 'Accion', 'Flujo_Neto', 'Punto_Reorden', 'CV', 'ADU', 'Num_Outliers_Ajustados']
        cols_resto = [c for c in df_master.columns if c not in cols_prioridad]
        df_experto = df_master[cols_prioridad + cols_resto]

        df_experto.to_excel(writer, sheet_name='Analisis Experto', index=False)

        # Alerta visual para Outliers en hoja experto
        ws_e = writer.sheets['Analisis Experto']
        fmt_warn = workbook.add_format({'bg_color': '#FFFF00', 'bold': True})
        idx_outliers = list(df_experto.columns).index('Num_Outliers_Ajustados')
        ws_e.conditional_format(1, idx_outliers, len(df_experto), idx_outliers,
                               {'type': 'cell', 'criteria': '>', 'value': 0, 'format': fmt_warn})

        # --- HOJA 3: LOG OUTLIERS ---
        if df_log_outliers is not None and not df_log_outliers.empty:
            df_log_outliers.to_excel(writer, sheet_name='Log Outliers', index=False)

        writer.close()
        print(">>> PROCESO COMPLETADO EXITOSAMENTE <<<")

    except Exception as e:
        print(f"Error al guardar Excel: {e}")
        # Intento de guardar sin formato si falla XlsxWriter
        df_master.to_excel("Backup_Sin_Formato.xlsx")

# ==============================================================================
# EJECUCIÓN DEL SCRIPT
# ==============================================================================
if __name__ == "__main__":
    # 1. Cargar
    hist, inv, erp = cargar_datos_reales()

    # 2. Procesar (Solo si cargó bien)
    if hist is not None and inv is not None:
        resultado, logs = motor_abastecimiento(hist, inv, erp)

        # 3. Exportar
        generar_excel_completo(resultado, logs)
    else:
        print("No se pudo completar el proceso por errores en la carga de archivos.")

--- INICIANDO PROCESO DE CARGA ---
Cargando Histórico...
Cargando Inventario...
Cargando Reporte ERP...
¡Datos cargados correctamente!
Ejecutando algoritmos de optimización...
Generando archivo: Plan_Abastecimiento_2026-07-15.xlsx ...
>>> PROCESO COMPLETADO EXITOSAMENTE <<<
